# Clerk with a Landscape, Clustergram and Yearbook: Atera human colorectal cancer

Scratch notebook for trying Clerk on the Atera colorectal cancer tutorial data (not part of the docs). It uses the same public data as the [tutorial](https://broadinstitute.github.io/celldega/examples/tutorial_notebooks/Atera_Colorectal_Cancer_Landscape-Matrix/): DegaFiles on GitHub and the pre-clustered Matrix on Hugging Face.

Layout: a Landscape and a Clustergram linked to each other and to Clerk in one row, then a **Yearbook** below, also linked to Clerk. Each question sends Clerk what all three show (the Yearbook's portraits too), and Clerk can change any of them.

What to try:

1. **Ask about the views.** Select a cluster or genes, then ask Clerk about them. Enrichr results are **cached**: in this conversation Clerk reuses lookups it already made, and across sessions they're read from disk (`clerk_files/lookup_cache/`) instead of asking Enrichr again.
2. **Look at single cells.** Ask Clerk to show you cells in the Yearbook, e.g. a cluster's cells ranked by one of its markers, or cells chosen by any combination of cluster and genes (see *Selecting Yearbook cells* below).
3. **Save and reopen.** Click **Report** (or run `clerk.export()`), then reopen it with `clerk.load_conversation(...)` at the end of this notebook.

In [1]:
import os

# Use this checkout's JS build (npm run build / npm run watch), served once over HTTP
# so each widget stores a tiny import instead of the whole bundle. After a rebuild,
# restart the kernel and reload the page. If widgets fail to load (e.g. a remote
# kernel), use "1" instead to inline the bundle. Set before importing celldega.
os.environ["CELLDEGA_LOCAL_ESM"] = "serve"

In [2]:
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
import tempfile
from urllib.request import urlopen

import anndata as ad
import celldega as dega
import numpy as np
import pandas as pd
import scipy.sparse as sp
import zarr

print(f"celldega {version('celldega')}")

SAMPLE = "Atera_v1_WTA_S4B_Colorectal_Cancer_FF"
HF_BASE = "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/Atera"
# DegaFiles for the Landscape and Yearbook: GitHub (one file per tile)
BASE_URL = "https://raw.githubusercontent.com/broadinstitute/celldega_atera-wta-ff-human-crc/main"
# The Matrix (pre-clustered AnnData-Zarr) is hosted on Hugging Face only.
MATRIX_URL = f"{HF_BASE}/{SAMPLE}/cgm/default_clusters"
# Downloads are kept here, so re-running the notebook doesn't fetch them again.
DATA_DIR = Path(tempfile.gettempdir()) / "celldega_atera" / SAMPLE
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Atera centroids are in microns; this matrix maps microns to image pixels.
with urlopen(f"{BASE_URL}/micron_to_image_transform.csv") as response:
    transform = np.loadtxt(response)

celldega 0.28.0a1


## Log in to Claude (one time)

Clerk sends questions to Claude through the `claude` command-line tool (Claude Code), with your own Claude account. If you haven't yet, run `claude auth login` in a terminal on the machine running this kernel. Celldega never sees your credentials.

In [3]:
status = dega.clerk.claude_auth_status()
if status["logged_in"]:
    plan = f", {status['subscription_type']} plan" if status["subscription_type"] else ""
    print(f"Logged in to Claude ({status['auth_method']}{plan}).")
else:
    print("Not logged in yet:", status.get("error") or "run `claude auth login` in a terminal.")

Logged in to Claude (claude.ai, enterprise plan).


## Load the pre-clustered Matrix

The default-cluster Matrix (pseudo-bulk signatures of every cell, dot sizes, and marker ranks from `scanpy.tl.rank_genes_groups`) was computed once and saved as an AnnData-Zarr. It's downloaded once (a few MB) and read with `Matrix.read_zarr`; nothing is re-clustered.

In [4]:
local = DATA_DIR / "default_clusters.zarr.zip"
if not local.is_file():
    with urlopen(f"{MATRIX_URL}.zarr.zip") as response:
        local.write_bytes(response.read())

with zarr.storage.ZipStore(str(local), mode="r") as store:
    matrix = dega.clust.Matrix.read_zarr(store)

print(f"{matrix.to_df().shape[1]} clusters x {matrix.to_df().shape[0]:,} genes")

24 clusters x 24,677 genes


## A single-cell AnnData for selecting Yearbook cells

Clerk can show cells in the Yearbook in two ways:

- **The Yearbook's own query**: a cluster's cells, ranked by a gene's expression. The Yearbook runs it in the browser, so no AnnData is needed.
- **A `celldega.select` selection**: any combination of cell attributes and genes (e.g. cells of one cluster that express one gene but not another, ranked by a third). Clerk writes the query and sampler as JSON; the Clerk widget runs them in this kernel against an AnnData you give it. Clerk runs no code itself.

For the second, Clerk needs a single-cell AnnData whose cell names match the Yearbook's. This dataset has no published one, so the cell below builds a light one from the DegaFiles: every cell, its cluster and position, and raw counts of the Clustergram's top 3 markers per cluster (about 60 genes, an ~80 MB download the first time, then cached as an `.h5ad`).

Clerk takes it as an **AnnData link** (a path or URL to an `.h5ad` or `.zarr`), reads it the first time it needs it, and tells the analysis notebooks it drafts where the data is. An in-memory AnnData works too.

In [5]:
ADATA_PATH = DATA_DIR / "cells_top_markers.h5ad"


def read_parquet(path):
    with urlopen(f"{BASE_URL}/{path}") as response:
        return pd.read_parquet(BytesIO(response.read()))


if not ADATA_PATH.is_file():
    # The Clustergram's top 3 markers per cluster (Wilcoxon ranks stored in the Matrix).
    ranks = pd.DataFrame(matrix.to_clustered_adata().uns["marker_ranks"])
    genes = list(dict.fromkeys(ranks.loc[ranks["rank"] < 3].sort_values(["rank", "group"])["names"]))

    cells = read_parquet("cell_metadata.parquet")  # name, [x, y] in microns
    clusters = read_parquet("cell_clusters/cluster.parquet")["cluster"]
    obs = pd.DataFrame(index=pd.Index(cells["name"].astype(str)))
    # Cells without a default cluster stay missing (not the text "<NA>").
    obs["cluster"] = pd.Categorical(clusters.reindex(obs.index).astype("string"))

    # cbg/<gene>.parquet: one gene's counts in the cells that have any, indexed by
    # the cell's row in cell_metadata.
    with ThreadPoolExecutor(8) as pool:
        columns = list(pool.map(lambda gene: read_parquet(f"cbg/{gene}.parquet"), genes))
    rows = np.concatenate([column.index.to_numpy() for column in columns])
    cols = np.concatenate([np.full(len(column), j) for j, column in enumerate(columns)])
    counts = np.concatenate([column.iloc[:, 0].to_numpy("float32") for column in columns])
    X = sp.csr_matrix((counts, (rows, cols)), shape=(len(obs), len(genes)))

    cells_adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=genes))
    cells_adata.obsm["spatial"] = np.array(cells["geometry"].tolist(), dtype="float32")
    cells_adata.uns["note"] = "Raw counts of the Clustergram's top 3 markers per cluster only."
    cells_adata.write_h5ad(ADATA_PATH)

print(ad.read_h5ad(ADATA_PATH, backed="r"))

AnnData object with n_obs × n_vars = 446777 × 51 backed at '/var/folders/8d/jxpy9rd10j7fp2rcj_s5sz3c0000gq/T/celldega_atera/Atera_v1_WTA_S4B_Colorectal_Cancer_FF/cells_top_markers.h5ad'
    obs: 'cluster'
    uns: 'note'
    obsm: 'spatial'


## Clerk, linked to a Landscape and the Clustergram

`spatial_clustergram(..., clerk=clerk)` links the Landscape and Clustergram to each other and to Clerk, with Clerk beside them. Click a gene row to show its expression, or a cluster column to select its cells, then ask Clerk about what you see.

**Enrichr cache:** Clerk keeps Enrichr (and PubMed) results in `<output_dir>/lookup_cache` (here `clerk_files/lookup_cache`), so a lookup it already made, in this session or an earlier one, isn't sent to Enrichr again. Within a conversation Clerk also sees its earlier Enrichr results and reuses them. Set `clerk.lookup_cache = None` to keep results for this session only.

In [6]:
clerk = dega.viz.Clerk(
    # An AnnData link (or an AnnData): lets Clerk select Yearbook cells with celldega.select.
    adata=str(ADATA_PATH),
    context_info=(
        "10x Genomics Atera whole-transcriptome imaging of human colorectal cancer "
        "(fresh frozen), with the dataset's default clusters."
    ),
)
print("Enrichr cache:", Path(clerk.output_dir) / clerk.lookup_cache)

clustergram = dega.viz.Clustergram(
    matrix=matrix, width=400, height=500, rank_dim=3,
    viz_mode="dotplot", manual_row_cat=False, manual_col_cat=False,
)
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera colorectal cancer — default clusters", height=600,
    max_tiles_to_view=25,
)
# A 520px Landscape (just wider than its controls) and a 400px Clustergram (a ~500px
# panel with its labels) leave room for Clerk (240-300px) in about 1,300px. A narrower
# notebook scrolls the row sideways; clerk_layout="wide" puts Clerk below instead.
linked = dega.viz.spatial_clustergram(landscape, clustergram, width="520px", clerk=clerk)

Enrichr cache: clerk_files/lookup_cache


In [7]:
linked

## Yearbook, linked to Clerk

A grid of single-cell portraits, rendered below the linked views. Linking it to Clerk sends its portraits (query, page, cells and an image) with each question, and lets Clerk choose which cells it shows.

This starts on cluster **3** ranked by **CLDN4**, as in the tutorial. Try asking Clerk to show you the cells behind something it said, e.g. a cluster's cells ranked by one of its markers, or cells of a cluster that express one marker but not another.

In [8]:
yearbook = dega.viz.Yearbook(
    base_url=BASE_URL,
    front_end_query={
        "cluster": {"attr": "cluster", "value": "3"},
        "gene": "CLDN4",
        "max_cells": 60,
    },
    rows=2, cols=6, portrait_size_um=50,
    dataset_name="Atera colorectal cancer — cell yearbook", width=0, height=520,
)
yearbook.layout.width = "100%"
dega.viz.link_clerk(clerk, yearbook=yearbook)
yearbook

### Selecting Yearbook cells yourself

What Clerk does when it selects cells, done by hand: a `celldega.select` query and sampler, and the selection given to the Yearbook. `query.to_dict()` is the JSON Clerk writes; `dega.select.query_from_dict` turns it back into a query.

In [ ]:
cells_adata = ad.read_h5ad(ADATA_PATH)
selector = dega.select.Selector(cells_adata)

# Cluster 3's top two markers, from the Matrix's stored ranks.
ranks = pd.DataFrame(matrix.to_clustered_adata().uns["marker_ranks"])
first, second = ranks[ranks["group"] == "3"].sort_values("rank")["names"].head(2)

# Cells of cluster 3 that express its second marker, ranked by its first.
query = (selector.attr("cluster") == "3") & (selector.gene(second) > 0)
selection = selector.select(
    query=query,
    sampler=selector.samplers.rank(selector.gene(first), n=24),
)
print(len(selection), "cells;", "query:", query.to_dict())
# Uncomment to show them in the Yearbook:
# yearbook.front_end_query = {}
# yearbook.selection = selection.to_json()
# yearbook.cells = selection.names()

## Save the report, then reopen it

**Report** in the Clerk panel (or `clerk.export()`) writes a folder in `clerk_files/`: `report.html`, `report.ipynb`, `record.json`, and every view Clerk saw as a Celldega screenshot (`screenshots/`: Landscape, Clustergram and Yearbook PNGs that carry the state to reopen them).

Ask Clerk a question or two first.

In [ ]:
report = clerk.export(title="Atera CRC with Clerk")
report_folder = report["record"].parent
report_folder

Reopen it in the panel: the conversation comes back as it was, with its genes, clusters, cited views and sent images clickable (they change the widgets above, which show the same data). A new question continues the conversation, with the reopened one as its history.

The same works after restarting the kernel: run the cells above to show the widgets again, then this one (or use **OPEN** in the panel and choose the folder's `record.json`).

In [ ]:
clerk.load_conversation(report_folder)